## Embedings

Katika mfano wetu wa awali, tulifanya kazi na vekta za mita nyingi za mfuko-wa-maneno zenye urefu wa `vocab_size`, na tuliwa badilisha waziwazi kutoka kwa vekta za uwakilishi wa nafasi za n-dimensional kuwa uwakilishi mwafaka wa hoti-moja. Uwikilishi huu wa hoti-moja hauitaji kumbukumbu nyingi, pia neno kila moja linaloachwa bila kuunganishwa na lingine, yaani vekta zilizowekwa kwa hoti-moja hazioneshi uhusiano wa maana kati ya maneno.

Katika kitengo hiki, tutaendelea kuchunguza dataset ya **News AG**. Ili kuanza, hebu paku data na tupate baadhi ya tafsiri kutoka kwa daftari la awali.


In [1]:
import torch
import torchtext
import numpy as np
from torchnlp import *
train_dataset, test_dataset, classes, vocab = load_dataset()
vocab_size = len(vocab)
print("Vocab size = ",vocab_size)

Loading dataset...


d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\train.csv: 29.5MB [00:01, 18.8MB/s]                            
d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\test.csv: 1.86MB [00:00, 11.2MB/s]                          


Building vocab...
Vocab size =  95812



## Embedding ni nini?

Wazo la **embedding** ni kuwakilisha maneno kwa kutumia vekta za msongamano wa chini-dimensionali, ambazo kwa namna fulani zinaonyesha maana ya semantiki ya neno. Baadaye tutajadili jinsi ya kujenga embeddings za maneno zenye maana, lakini kwa sasa hebu fikiria embeddings kama njia ya kupunguza dimensionali ya vekta ya neno.

Kwa hivyo, safu ya embedding ingechukua neno kama ingizo, na kutoa vekta ya pato yenye `embedding_size` kama ilivyoainishwa. Kwa maana fulani, ni sawa sana na safu ya `Linear`, lakini badala ya kuchukua vekta ya one-hot encoded, itakuwa na uwezo wa kuchukua nambari ya neno kama ingizo.

Kwa kutumia safu ya embedding kama safu ya kwanza katika mtandao wetu, tunaweza kubadili kutoka mfuko-wa-maneno hadi mfano wa **embedding bag**, ambapo kwanza tunageuza kila neno katika maandishi yetu kuwa embedding inayolingana, kisha tukahesabu kazi ya jumla juu ya embeddings zote hizo, kama vile `sum`, `average` au `max`.

![Picha inayoonyesha mzalishaji wa embeddings kwa maneno matano ya mfuatano.](../../../../../translated_images/sw/embedding-classifier-example.b77f021a7ee67eee.webp)

Mtandao wetu wa neva wa mzalishaji utaanza na safu ya embedding, kisha safu ya kujumlisha, na mzalisaji laini juu yake:


In [2]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, x):
        x = self.embedding(x)
        x = torch.mean(x,dim=1)
        return self.fc(x)

### Kuboresha ukubwa wa mfululizo wa vigezo

Kama matokeo ya usanifu huu, minibatches kwa mtandao wetu itabidi izalishwe kwa njia fulani. Katika kitengo kilichopita, wakati wa kutumia mfuko-wa-manikani, tensors zote za BoW katika minibatch zilikuwa na ukubwa sawa `vocab_size`, bila kujali urefu halisi wa mfululizo wetu wa maandishi. Mara tu tunapokuwa tumesogea kwenye embeddings za maneno, tutapata idadi inayobadilika ya maneno katika kila sampuli ya maandishi, na tunapoziunganisha sampuli hizo katika minibatches tutalazimika kutumia padding.

Hii inaweza kufanywa kwa kutumia mbinu ile ile ya kutoa kazi ya `collate_fn` kwa chanzo cha data:


In [3]:
def padify(b):
    # b is the list of tuples of length batch_size
    #   - first element of a tuple = label, 
    #   - second = feature (text sequence)
    # build vectorized sequence
    v = [encode(x[1]) for x in b]
    # first, compute max length of a sequence in this minibatch
    l = max(map(len,v))
    return ( # tuple of two tensors - labels and features
        torch.LongTensor([t[0]-1 for t in b]),
        torch.stack([torch.nn.functional.pad(torch.tensor(t),(0,l-len(t)),mode='constant',value=0) for t in v])
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=padify, shuffle=True)

### Kufundisha kisafirishaji cha embedding

Sasa tunapokuwa tumetambua dataloader sahihi, tunaweza kufundisha modeli kwa kutumia kazi ya mafunzo tuliyoitambua katika kitengo kilichopita:


In [4]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)
train_epoch(net,train_loader, lr=1, epoch_size=25000)

3200: acc=0.6415625
6400: acc=0.6865625
9600: acc=0.7103125
12800: acc=0.726953125
16000: acc=0.739375
19200: acc=0.75046875
22400: acc=0.7572321428571429


(0.889799795315499, 0.7623160588611644)

> **Kumbuka**: Hapa tunafunza rekodi 25k tu (chini ya kipindi kimoja kamili) kwa ajili ya kuokoa muda, lakini unaweza kuendelea kufunza, andika kazi ya kufunza kwa vipindi vingi, na jaribu kipimo cha kasi ya kujifunza ili kufikia usahihi wa juu. Unapaswa kuwa na uwezo wa kufikia usahihi wa karibu 90%.


### Tabaka la EmbeddingBag na Uwasilishaji wa Mfululizo wa Urefu wa Tofauti

Katika usanifu uliopita, tulihitaji kujaza mfululizo wote hadi urefu sawa ili kuziweka katika minibatch. Hii siyo njia bora zaidi ya kuwasilisha mfululizo wa urefu tofauti - njia nyingine ingekuwa kutumia vekta ya **offset**, ambayo ingeweka offsets za mfululizo wote ulihifadhiwa katika vekta kubwa moja.

![Picha inayoonyesha uwasilishaji wa mfululizo wa offset](../../../../../translated_images/sw/offset-sequence-representation.eb73fcefb29b46ee.webp)

> **Kumbuka**: Katika picha hapo juu, tunaonyesha mfululizo wa herufi, lakini katika mfano wetu tunafanya kazi na mfululizo wa maneno. Hata hivyo, kanuni ya jumla ya kuwasilisha mfululizo kwa vekta ya offset inabaki sawa.

Ili kufanya kazi na uwasilishaji wa offset, tunatumia tabaka la [`EmbeddingBag`](https://pytorch.org/docs/stable/generated/torch.nn.EmbeddingBag.html). Linafanana na `Embedding`, lakini linachukua vekta ya maudhui na vekta ya offset kama ingizo, na pia linajumuisha tabaka la wastani, ambalo linaweza kuwa `mean`, `sum` au `max`.

Hapa kuna mtandao uliobadilishwa unaotumia `EmbeddingBag`:


In [5]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.EmbeddingBag(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, text, off):
        x = self.embedding(text, off)
        return self.fc(x)

Kuandaa seti ya data kwa ajili ya mafunzo, tunahitaji kutoa kazi ya uongofu ambayo itaandaa vekta ya msogeo:


In [6]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1])) for t in b]
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)

Kumbuka, kwamba tofauti na mifano yote iliyotangulia, mtandao wetu sasa unakubali vigezo viwili: vekta ya data na vekta ya ofseti, ambavyo vina ukubwa tofauti. Vilevile, mzigo wetu wa data hutupatia thamani 3 badala ya 2: vekta zote za maandishi na ofseti hutolewa kama sifa. Kwa hivyo, tunahitaji kurekebisha kidogo kazi yetu ya mafunzo ili kushughulikia hilo:


In [7]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)

def train_epoch_emb(net,dataloader,lr=0.01,optimizer=None,loss_fn = torch.nn.CrossEntropyLoss(),epoch_size=None, report_freq=200):
    optimizer = optimizer or torch.optim.Adam(net.parameters(),lr=lr)
    loss_fn = loss_fn.to(device)
    net.train()
    total_loss,acc,count,i = 0,0,0,0
    for labels,text,off in dataloader:
        optimizer.zero_grad()
        labels,text,off = labels.to(device), text.to(device), off.to(device)
        out = net(text, off)
        loss = loss_fn(out,labels) #cross_entropy(out,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss
        _,predicted = torch.max(out,1)
        acc+=(predicted==labels).sum()
        count+=len(labels)
        i+=1
        if i%report_freq==0:
            print(f"{count}: acc={acc.item()/count}")
        if epoch_size and count>epoch_size:
            break
    return total_loss.item()/count, acc.item()/count


train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6153125
6400: acc=0.6615625
9600: acc=0.6932291666666667
12800: acc=0.715078125
16000: acc=0.7270625
19200: acc=0.7382291666666667
22400: acc=0.7486160714285715


(22.771553103007037, 0.7551983365323096)


## Ufichuzi wa Kisemantiki: Word2Vec

Katika mfano wetu uliopita, safu ya ufichuzi wa modeli ilijifunza kupangilia maneno kwa mwakilishi wa vector, hata hivyo, mwakilishi huu hakuwa na maana nyingi kisemantiki. Itakuwa nzuri kujifunza mwakilishi wa vector kama huo, ambapo maneno yanayofanana au maneno ya maana moja yataendana na vector ambazo ziko karibu kwa upande wa umbali wa vector (mfano umbali wa euclidean).

Ili kufanikisha hilo, tunahitaji kufundisha awali mfano wetu wa ufichuzi kwenye mkusanyiko mkubwa wa maandishi kwa njia maalum. Moja ya njia za kwanza za kufundisha ufichuzi wa kisemantiki huitwa [Word2Vec](https://en.wikipedia.org/wiki/Word2vec). Inategemea miundo miwili kuu inayotumika kuunda mwakilishi wa maneno wa kusambazwa:

 - **Mfuko endelevu wa maneno** (CBoW) — kwenye muundo huu, tunafundisha modeli kutabiri neno kutoka muktadha wa karibu. Tukipata ngram $(W_{-2},W_{-1},W_0,W_1,W_2)$, lengo la modeli ni kutabiri $W_0$ kutoka $(W_{-2},W_{-1},W_1,W_2)$.
 - **Skip-gram endelevu** ni kinyume cha CBoW. Modeli hutumia dirisha la muktadha wa maneno ya karibu kutabiri neno la sasa.

CBoW ni haraka zaidi, wakati skip-gram ni polepole, lakini hufanya kazi bora ya kuwakilisha maneno yasiyotokea mara kwa mara.

![Picha inayoonyesha algorithms za CBoW na Skip-Gram kubadilisha maneno kuwa vectors.](../../../../../translated_images/sw/example-algorithms-for-converting-words-to-vectors.fbe9207a726922f6.webp)

Ili kujaribu ufichuzi wa word2vec uliotanguliwa na dataset ya Google News, tunaweza kutumia maktaba ya **gensim**. Hapa chini tunapata maneno yanayofanana zaidi na 'neural'

> **Kumbuka:** Unapounda vectors za maneno kwa mara ya kwanza, kuzipakua kunaweza kuchukua wakati kidogo!


In [8]:
import gensim.downloader as api
w2v = api.load('word2vec-google-news-300')

In [9]:
for w,p in w2v.most_similar('neural'):
    print(f"{w} -> {p}")

neuronal -> 0.7804799675941467
neurons -> 0.7326500415802002
neural_circuits -> 0.7252851724624634
neuron -> 0.7174385190010071
cortical -> 0.6941086649894714
brain_circuitry -> 0.6923246383666992
synaptic -> 0.6699118614196777
neural_circuitry -> 0.6638563275337219
neurochemical -> 0.6555314064025879
neuronal_activity -> 0.6531826257705688


Pia tunaweza kuhesabu uingizaji vekta kutoka kwa neno, utakaotumika katika mafunzo ya modeli ya uainishaji (tunaonyesha tu vipengele 20 vya kwanza vya vekta kwa uwazi):


In [10]:
w2v.word_vec('play')[:20]

array([ 0.01226807,  0.06225586,  0.10693359,  0.05810547,  0.23828125,
        0.03686523,  0.05151367, -0.20703125,  0.01989746,  0.10058594,
       -0.03759766, -0.1015625 , -0.15820312, -0.08105469, -0.0390625 ,
       -0.05053711,  0.16015625,  0.2578125 ,  0.10058594, -0.25976562],
      dtype=float32)

Jambo kubwa kuhusu embeds za semantiki ni kwamba unaweza kudhibiti usimbaji wa vekta kubadilisha semantiki. Kwa mfano, tunaweza kuomba kupata neno, ambalo usambazaji wake wa vekta ungekuwa wa karibu sana na maneno *king* na *woman*, na wa mbali iwezekanavyo na neno *man*:


In [10]:
w2v.most_similar(positive=['king','woman'],negative=['man'])[0]

('queen', 0.7118192911148071)

CBoW na Skip-Grams zote ni embedding za “utabiri”, kwa maana wanachukua tu muktadha wa ndani kuzingatia. Word2Vec haichukui faida ya muktadha wa ulimwengu mzima. 

**FastText**, inajengwa juu ya Word2Vec kwa kujifunza uwakilishi wa vekta kwa kila neno na n-grams za herufi zilizopo ndani ya kila neno. Thamani za uwakilishi hizi zinaelekezwa kuwa vekta moja katika kila hatua ya mafunzo. Ingawa hii inaongeza mahesabu mengi kabla ya mafunzo, inaruhusu embedding za maneno kuwa na taarifa za chini ya neno. 

Njia nyingine, **GloVe**, inatumia wazo la matriki ya kuwepo pamoja, inatumia njia za neural kuvunjua matriki ya kuwepo pamoja kuwa vekta za maneno zenye hisia zaidi na zisizo za mstari.

Unaweza kucheza na mfano kwa kubadilisha embedding kuwa FastText na GloVe, kwa kuwa gensim inaunga mkono mifano kadhaa tofauti ya embedding za maneno.


## Kutumia Mwekundu wa Awali Uliofunzwa katika PyTorch

Tunaweza kubadilisha mfano ulio hapo juu ili kujaza awali matriki katika tabaka letu la wekundu na wekundu wa maana, kama vile Word2Vec. Tunahitaji kuzingatia kwamba msamiati wa wekundu uliowekwa awali na mkusanyiko wetu wa maandishi huenda hauendani, kwa hivyo tutaanzisha uzito wa maneno yaliyokosekana kwa kutumia thamani za nasibu:


In [11]:
embed_size = len(w2v.get_vector('hello'))
print(f'Embedding size: {embed_size}')

net = EmbedClassifier(vocab_size,embed_size,len(classes))

print('Populating matrix, this will take some time...',end='')
found, not_found = 0,0
for i,w in enumerate(vocab.get_itos()):
    try:
        net.embedding.weight[i].data = torch.tensor(w2v.get_vector(w))
        found+=1
    except:
        net.embedding.weight[i].data = torch.normal(0.0,1.0,(embed_size,))
        not_found+=1

print(f"Done, found {found} words, {not_found} words missing")
net = net.to(device)

Embedding size: 300
Populating matrix, this will take some time...Done, found 41080 words, 54732 words missing


Sasa hebu tufunze mfano wetu. Kumbuka kwamba muda unaochukua kufundisha mfano ni mkubwa zaidi ikilinganishwa na mfano wa awali, kutokana na ukubwa mkubwa wa safu ya usisianisho, na hivyo idadi kubwa zaidi ya vigezo. Pia, kwa sababu hii, tunaweza kuhitaji kufundisha mfano wetu kwa mifano zaidi ikiwa tunataka kuepuka kufundisha kupita kiasi.


In [12]:
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6359375
6400: acc=0.68109375
9600: acc=0.7067708333333333
12800: acc=0.723671875
16000: acc=0.73625
19200: acc=0.7463541666666667
22400: acc=0.7560714285714286


(214.1013875559821, 0.7626759436980166)

Katika kesi yetu hatuoni ongezeko kubwa la usahihi, jambo ambalo huenda likatokana na msamiati tofauti kabisa. 
Ili kushinda tatizo la misamiati tofauti, tunaweza kutumia moja ya suluhisho zifuatazo:
* Funga upya modeli ya word2vec kwenye msamiati wetu
* Pakia seti yetu ya data na msamiati kutoka kwenye modeli ya word2vec iliyofunzwa awali. Msamiati unaotumika kupakia seti ya data unaweza kuelezwa wakati wa kupakia.

Njia ya mwisho inaonekana rahisi zaidi, hasa kwa sababu mfumo wa PyTorch `torchtext` una msaada uliojengwa ndani kwa embeddings. Tunaweza, kwa mfano, kuanzisha msamiati unaotokana na GloVe kwa njia ifuatayo:


In [14]:
vocab = torchtext.vocab.GloVe(name='6B', dim=50)

100%|█████████▉| 399999/400000 [00:15<00:00, 25411.14it/s]


Msamiati uliopakiwa una shughuli msingi zifuatazo:
* `vocab.stoi` kamusi huturuhusu kubadilisha neno kuwa index yake ya kamusi
* `vocab.itos` hufanya kinyume - hubadilisha nambari kuwa neno
* `vocab.vectors` ni safu ya vekta za kuingiza, kwa hivyo kupata ujumuishaji wa neno `s` tunahitaji kutumia `vocab.vectors[vocab.stoi[s]]`

Hapa kuna mfano wa kushughulikia ujumuishaji kuonyesha mlinganyo **mpole-mwanaume+mwanamke = malkia** (nililazimika kurekebisha kidogo mgawo ili kufanya ifanye kazi):


In [15]:
# get the vector corresponding to kind-man+woman
qvec = vocab.vectors[vocab.stoi['king']]-vocab.vectors[vocab.stoi['man']]+1.3*vocab.vectors[vocab.stoi['woman']]
# find the index of the closest embedding vector 
d = torch.sum((vocab.vectors-qvec)**2,dim=1)
min_idx = torch.argmin(d)
# find the corresponding word
vocab.itos[min_idx]

'queen'

Ili kufundisha m分類Kwa kutumia maumbo yale, kwanza tunahitaji kuandika seti yetu ya data kwa kutumia msamiati wa GloVe:


In [16]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1],voc=vocab)) for t in b] # pass the instance of vocab to encode function!
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

Kama tulivyoona hapo juu, vionjo vyote vya vector vinahifadhiwa katika matriki ya `vocab.vectors`. Hii inafanya iwe rahisi sana kupakia uzito huo kuwa uzito wa tabaka la embedding kwa kutumia kunakili rahisi:


In [17]:
net = EmbedClassifier(len(vocab),len(vocab.vectors[0]),len(classes))
net.embedding.weight.data = vocab.vectors
net = net.to(device)

Sasa wacha tufunze modeli yetu na tutaone kama tuna matokeo bora zaidi:


In [18]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6271875
6400: acc=0.68078125
9600: acc=0.7030208333333333
12800: acc=0.71984375
16000: acc=0.7346875
19200: acc=0.7455729166666667
22400: acc=0.7529464285714286


(35.53972978646833, 0.7575175943698017)

Moja ya sababu tunazoshindwa kuona ongezeko kubwa la usahihi ni kwa sababu baadhi ya maneno kutoka kwenye dataset yetu hayapo kwenye msamiati wa GloVe uliolindwa tayari, na kwa hiyo kimsingi hayajaliwi. Ili kushinda tatizo hili, tunaweza kufundisha embeddings zetu wenyewe kwenye dataset yetu.


## Embeddings za Muktadha

Kizuizi kikuu cha kawaida cha mawakilishi ya embeddings yaliyopangwa awali kama Word2Vec ni tatizo la ufafanuzi wa maana ya neno. Wakati embeddings zilizopangwa awali zinaweza kunasa baadhi ya maana za maneno katika muktadha, kila maana inayowezekana ya neno inaandikwa kwenye embedding ile ile. Hii inaweza kusababisha matatizo katika modeli zinazofuata, kwani maneno mengi kama neno 'play' yana maana tofauti kulingana na muktadha yanapotumika.

Kwa mfano, neno 'play' katika sentensi mbili tofauti zina maana tofauti kabisa:
- Nilikwenda kwenye **play** katika theatre.
- John anataka **play** na marafiki zake.

Embeddings zilizopangwa awali hapo juu zinawakilisha maana zote hizi za neno 'play' katika embedding ile ile. Ili kushinda kizuizi hiki, tunahitaji kujenga embeddings kulingana na **mfano wa lugha**, ambao umefundishwa kwenye korasi kubwa ya maandishi, na *unajua* jinsi maneno yanavyoweza kuwekwa pamoja katika muktadha tofauti. Kujadili embeddings za muktadha ni nje ya upeo wa mafunzo haya, lakini tutarudi kwenye mada hii tunapozungumzia kuhusu mifano ya lugha kwenye kitengo kijacho.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Kionyozo**:
Hati hii imetafsiriwa kwa kutumia huduma ya tafsiri ya AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kupata usahihi, tafadhali fahamu kwamba tafsiri za kiotomatiki zinaweza kuwa na makosa au upungufu wa usahihi. Hati ya asili katika lugha yake halisi inapaswa kuchukuliwa kama chanzo cha mamlaka. Kwa taarifa muhimu, tafsiri ya kitaalamu inayofanywa na binadamu inapendekezwa. Hatutojibu kwa kuelewa vibaya au tafsiri potofu zinazotokea kutokana na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
